In [1]:
import h5py
import pandas as pd
from autogluon.tabular import TabularPredictor

def predict_and_save_top(
    model_path,
    target_name,
    library_df,
    top_n=10,
    output_csv_path=None
):
    """
    Loads an AutoGluon TabularPredictor, predicts on library_df, 
    sorts by predicted values, and saves/returns top predictions.
    
    Parameters
    ----------
    model_path : str
        Path to the AutoGluon model directory.
    target_name : str
        The prediction target name (used as column label in results).
    library_df : pd.DataFrame
        DataFrame containing the features for prediction.
    top_n : int, optional
        Number of top results to return (default 10).
    output_csv_path : str, optional
        If provided, saves the full prediction table to this CSV path.
    
    Returns
    -------
    top_datasets : list
        List of dataset names for the top_n predictions.
    top_df : pd.DataFrame
        DataFrame of the top_n predictions.
    """
    # Load model
    predictor = TabularPredictor.load(model_path)
    
    # Predict
    preds = predictor.predict(library_df)
    
    # Attach predictions to DataFrame
    results_df = library_df.copy()
    results_df[target_name] = preds
    
    # Sort and select top N
    sorted_df = results_df.sort_values(by=target_name, ascending=False).reset_index(drop=True)
    # Select top N or full dataframe
    if top_n is None:
        top_df = sorted_df[["Dataset", target_name]].copy()
    else:
        top_df = sorted_df[["Dataset", target_name]].head(top_n).copy()
    top_datasets = top_df["Dataset"].tolist()
    
    # Save if path provided
    if output_csv_path:
        sorted_df[["Dataset", target_name]].to_csv(output_csv_path, index=False)
    
    return top_datasets, top_df


In [2]:
# --- Load CSV files into lists ---
full_library_paths = [
    "Featurized datasets/Full Library/Full Library Dataset 1.csv",
    "Featurized datasets/Full Library/Full Library Dataset 2.csv",
    "Featurized datasets/Full Library/Full Library Dataset 3.csv",
    "Featurized datasets/Full Library/Full Library Dataset 4.csv",
    "Featurized datasets/Full Library/Full Library Dataset 5.csv"
]

full_library_OHE_path = "Featurized datasets/Full Library/Full Library one-hot encoded.csv"

full_library_dfs = [pd.read_csv(p) for p in full_library_paths]
full_library_OHE_df = pd.read_csv(full_library_OHE_path)

# Load dataset names from the HDF5 file
training_h5 = "HDF5 Datasets/250708 - Training Dataset.h5"
with h5py.File(training_h5, "r") as f:
    h5_dataset_names = list(f.keys())

# Filtering function
def filter_training_data(df, dataset_col="Dataset"):
    # Keep only rows ending with "_B01"
    df = df[df[dataset_col].str.endswith("_B01", na=False)]
    # Remove datasets already in HDF5 training data
    return df[~df[dataset_col].isin(h5_dataset_names)]


# Apply filtering to all datasets in lists
full_library_dfs = [filter_training_data(df) for df in full_library_dfs]
full_library_OHE_df = filter_training_data(full_library_OHE_df)


In [3]:
# Load models
model_path_CO_ds1 = "Autogluon Models/CO_tbu/Dataset 1/best_quality/CO_tbu_Dataset 1_best_quality"
model_path_CO_ds2 = "Autogluon Models/CO_tbu/Dataset 2/best_quality/CO_tbu_Dataset 2_best_quality"
model_path_CO_ds3 = "Autogluon Models/CO_tbu/Dataset 3/best_quality/CO_tbu_Dataset 3_best_quality"
model_path_CO_ds4 = "Autogluon Models/CO_tbu/Dataset 4/best_quality/CO_tbu_Dataset 4_best_quality"
model_path_CO_ds5 = "Autogluon Models/CO_tbu/Dataset 5/best_quality/CO_tbu_Dataset 5_best_quality"

model_path_MP_ds1 = "Autogluon Models/MP_CF3/Dataset 1/best_quality/MP_CF3_Dataset 1_best_quality"
model_path_MP_ds2 = "Autogluon Models/MP_CF3/Dataset 2/best_quality/MP_CF3_Dataset 2_best_quality"
model_path_MP_ds3 = "Autogluon Models/MP_CF3/Dataset 3/best_quality/MP_CF3_Dataset 3_best_quality"
model_path_MP_ds4 = "Autogluon Models/MP_CF3/Dataset 4/best_quality/MP_CF3_Dataset 4_best_quality"
model_path_MP_ds5 = "Autogluon Models/MP_CF3/Dataset 5/best_quality/MP_CF3_Dataset 5_best_quality"

model_path_CO_OHE = "Autogluon Models/One Hot Encoding/CO_tbu/One Hot Encoding/best_quality/CO_tbu_One Hot Encoding_best_quality"
model_path_MP_OHE = "Autogluon Models/One Hot Encoding/MP_CF3/One Hot Encoding/best_quality/MP_CF3_One Hot Encoding_best_quality"

# Predicting on  library 

In [4]:
# Train a small dummy regression model with AutoGluon

# Create a small dummy dataset
dummy_df = pd.DataFrame({
    'feature1': [1, 2, 3, 4],
    'feature2': [10, 20, 30, 40],
    'target': [1, 2, 3, 4]
})

# Train the dummy model
dummy_predictor = TabularPredictor(label='target', path = 'Autogluon Models/dummy model', eval_metric='r2').fit(
    train_data=dummy_df,
    presets='medium_quality',
    verbosity=0
)

		Expected n_neighbors <= n_samples_fit, but n_neighbors = 5, n_samples_fit = 2, n_samples = 2
		Expected n_neighbors <= n_samples_fit, but n_neighbors = 5, n_samples_fit = 2, n_samples = 2


In [5]:
# (model_path, dataframe)
prediction_jobs = {

    # CO_tbu models
    "CO_ds1": (model_path_CO_ds1, full_library_dfs[0]),
    "CO_ds2": (model_path_CO_ds2, full_library_dfs[1]),
    "CO_ds3": (model_path_CO_ds3, full_library_dfs[2]),
    "CO_ds4": (model_path_CO_ds4, full_library_dfs[3]),
    "CO_ds5": (model_path_CO_ds5, full_library_dfs[4]),

    # MP_CF3 models
    "MP_ds1": (model_path_MP_ds1, full_library_dfs[0]),
    "MP_ds2": (model_path_MP_ds2, full_library_dfs[1]),
    "MP_ds3": (model_path_MP_ds3, full_library_dfs[2]),
    "MP_ds4": (model_path_MP_ds4, full_library_dfs[3]),
    "MP_ds5": (model_path_MP_ds5, full_library_dfs[4]),
    
    # One-hot encoded
    "CO_OHE": (model_path_CO_OHE, full_library_OHE_df),
    "MP_OHE": (model_path_MP_OHE, full_library_OHE_df),

}

# --- Run predictions for all jobs ---
predictions = {}
prediction_dfs = {}

for name, (model_path, df) in prediction_jobs.items():
    print(f"Running predictions for {name}...")
    pred, pred_df = predict_and_save_top(
        model_path=model_path,
        target_name="CO_tbu" if "CO" in name else "MP_CF3",
        library_df=df,
        top_n=10
    )
    predictions[name] = pred
    prediction_dfs[name] = pred_df


Running predictions for CO_ds1...
Running predictions for CO_ds2...
Running predictions for CO_ds3...
Running predictions for CO_ds4...
Running predictions for CO_ds5...
Running predictions for MP_ds1...
Running predictions for MP_ds2...
Running predictions for MP_ds3...
Running predictions for MP_ds4...
Running predictions for MP_ds5...
Running predictions for CO_OHE...
Running predictions for MP_OHE...


In [6]:
print("CO arylation Top Predicted Performing Pre-stirring conditions")
print(predictions["CO_ds4"])
print("MP decarboxylative arylation Top Predicted Performing Pre-stirring conditions")
print(predictions["MP_ds4"])

CO arylation Top Predicted Performing Pre-stirring conditions
['M02_L07_S03_B01', 'M01_L35_S03_B01', 'M01_L07_S05_B01', 'M01_L35_S04_B01', 'M01_L45_S02_B01', 'M02_L38_S03_B01', 'M02_L41_S03_B01', 'M02_L35_S03_B01', 'M01_L45_S04_B01', 'M02_L01_S03_B01']
MP decarboxylative arylation Top Predicted Performing Pre-stirring conditions
['M02_L40_S03_B01', 'M02_L07_S03_B01', 'M02_L41_S03_B01', 'M01_L47_S04_B01', 'M02_L55_S03_B01', 'M02_L55_S04_B01', 'M02_L38_S03_B01', 'M02_L10_S03_B01', 'M02_L35_S03_B01', 'M01_L21_S04_B01']


In [7]:
# top CO ds4 predictions
pred, pred_df = predict_and_save_top(
    model_path=model_path_CO_ds4,
    target_name="CO_tbu",
    library_df=full_library_dfs[3],
    top_n=None
)
pred_df.to_csv("Autogluon Predictions/CO Dataset 4 predictions.csv")

In [8]:
# top CO OHE predictions
pred, pred_df = predict_and_save_top(
    model_path=model_path_CO_OHE,
    target_name="CO_tbu",
    library_df=full_library_OHE_df,
    top_n=None
)
pred_df.to_csv("Autogluon Predictions/CO OHE predictions.csv")

In [9]:
# top MP ds4 predictions
pred, pred_df = predict_and_save_top(
    model_path=model_path_MP_ds4,
    target_name="MP_CF3",
    library_df=full_library_dfs[3],
    top_n=None
)
pred_df.to_csv("Autogluon Predictions/MP Dataset 4 predictions.csv")

In [10]:
# top CO ds1 predictions
pred, pred_df = predict_and_save_top(
    model_path=model_path_CO_ds1,
    target_name="CO_CF3",
    library_df=full_library_dfs[0],
    top_n=None
)
pred_df.to_csv("Autogluon Predictions/CO Dataset 1 predictions.csv")

In [11]:
# top CO ds2 predictions
pred, pred_df = predict_and_save_top(
    model_path=model_path_CO_ds2,
    target_name="CO_tbu",
    library_df=full_library_dfs[1],
    top_n=None
)
pred_df.to_csv("Autogluon Predictions/CO Dataset 2 predictions.csv")

In [12]:
# top CO ds3 predictions
pred, pred_df = predict_and_save_top(
    model_path=model_path_CO_ds3,
    target_name="CO_tbu",
    library_df=full_library_dfs[2],
    top_n=None
)
pred_df.to_csv("Autogluon Predictions/CO Dataset 3 predictions.csv")

In [13]:
# top MP ds2 predictions
pred, pred_df = predict_and_save_top(
    model_path=model_path_MP_ds2,
    target_name="MP_CF3",
    library_df=full_library_dfs[1],
    top_n=None
)
pred_df.to_csv("Autogluon Predictions/MP Dataset 2 predictions.csv")

In [14]:
# top MP ds3 predictions
pred, pred_df = predict_and_save_top(
    model_path=model_path_MP_ds3,
    target_name="MP_CF3",
    library_df=full_library_dfs[2],
    top_n=None
)
pred_df.to_csv("Autogluon Predictions/MP Dataset 3 predictions.csv")

In [15]:
# top MP ds1 predictions
pred, pred_df = predict_and_save_top(
    model_path=model_path_MP_ds1,
    target_name="MP_CF3",
    library_df=full_library_dfs[0],
    top_n=None
)
pred_df.to_csv("Autogluon Predictions/MP Dataset 1 predictions.csv")

In [16]:
# top MP OHE predictions
pred, pred_df = predict_and_save_top(
    model_path=model_path_MP_OHE,
    target_name="MP_tbu",
    library_df=full_library_OHE_df,
    top_n=None
)
pred_df.to_csv("Autogluon Predictions/MP OHE predictions.csv")

# Saving predictions to CSV Files

In [8]:
# --- Load CSV files into lists ---
full_library_paths = [
    "Featurized datasets/Full Library/Full Library Dataset 4.csv",
]
full_library_dfs = [pd.read_csv(p) for p in full_library_paths]

model_path_CO_ds4 = "Autogluon Models/CO_tbu/Dataset 4/best_quality/CO_tbu_Dataset 4_best_quality"
model_path_MP_ds4 = "Autogluon Models/MP_CF3/Dataset 4/best_quality/MP_CF3_Dataset 4_best_quality"

pred, CO_full_pred_df = predict_and_save_top(
    model_path=model_path_CO_ds4,
    target_name="CO_tbu",
    library_df=full_library_dfs[0],
    top_n=None
)
CO_full_pred_df.to_csv("Autogluon Predictions/CO Dataset 4 predictions Full Library.csv")

pred, MP_full_pred_df = predict_and_save_top(
    model_path=model_path_MP_ds4,
    target_name="MP_CF3",
    library_df=full_library_dfs[0],
    top_n=None
)
MP_full_pred_df.to_csv("Autogluon Predictions/MP Dataset 4 predictions Full Library.csv")
